In [1]:
%pip install python-crfsuite

Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
from nltk.tag import CRFTagger
from nltk.corpus import treebank
from nltk import sent_tokenize, word_tokenize  

# Télécharger le corpus 
nltk.download('treebank', quiet=True)

# Charger les phrases taggées du Penn Treebank
# treebank.tagged_sents() renvoie ~3914 phrases (version NLTK standard)
all_sents = treebank.tagged_sents()

In [3]:

# Split simple 
split_idx = int(len(all_sents) * 0.8)
train_sents = all_sents[:split_idx]
test_sents  = all_sents[split_idx:]

print(f"Phrases d'entraînement : {len(train_sents)}")
print(f"Phrases de test        : {len(test_sents)}")


Phrases d'entraînement : 3131
Phrases de test        : 783


In [4]:
# Créer et entraîner le CRF tagger
ct = CRFTagger(verbose=True)  
ct.train(train_sents, 'crf_pos_tagger.model')

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 18393
Seconds required: 0.062

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 223112.798043
Feature norm: 5.000000
Error norm: 15534.697556
Active features: 18393
Line search trials: 2
Line search step: 0.000277
Seconds required for this iteration: 1.096

***** Iteration #2 *****
Loss: 213350.129528
Feature norm: 48.703556
Error norm: 15796.300156
Active features: 18393
Line search trials: 1
Line search step: 1.000000
Seconds required for this iteration: 0.352

***** Iteration #3 *****
Loss: 144816.519148
Feature norm: 54.662626
Error norm: 9755.506829
Active features: 18393
Line search trials: 1
Line search step: 1.000000
Seconds r

In [5]:
# Évaluer sur le test set
accuracy = ct.evaluate(test_sents)
print(f"\nAccuracy sur le test set : {accuracy:.4f} ")

C:\Users\afagn\AppData\Local\Temp\ipykernel_15176\2687491373.py:2: DeprecationWarning: 
  Function evaluate() has been deprecated.  Use accuracy(gold)
  instead.
  accuracy = ct.evaluate(test_sents)



Accuracy sur le test set : 0.9479 


In [6]:
from sklearn.metrics import classification_report
y_true = []
y_pred = []
print("Calcul des métriques par tag...")
for sent in test_sents:
    # On sépare les mots et les tags réels
    words = [w for w, t in sent]
    true_tags = [t for w, t in sent]
    
    # On prédit avec le modèle
    prediction = ct.tag(words)
    pred_tags = [t for w, t in prediction]
    
    # On accumule pour le rapport final
    y_true.extend(true_tags)
    y_pred.extend(pred_tags)
# Affichage du rapport complet
print(classification_report(y_true, y_pred))

Calcul des métriques par tag...
              precision    recall  f1-score   support

           #       1.00      1.00      1.00         2
           $       1.00      1.00      1.00       242
          ''       1.00      1.00      1.00        78
           ,       1.00      1.00      1.00       930
       -LRB-       1.00      1.00      1.00        26
      -NONE-       0.99      1.00      1.00      1340
       -RRB-       1.00      1.00      1.00        26
           .       1.00      1.00      1.00       762
           :       1.00      1.00      1.00        77
          CC       1.00      1.00      1.00       429
          CD       0.99      0.99      0.99      1032
          DT       0.99      0.99      0.99      1611
          EX       0.88      1.00      0.93         7
          IN       0.97      0.98      0.97      1952
          JJ       0.82      0.81      0.81      1087
         JJR       0.78      0.80      0.79        76
         JJS       0.83      0.89      0.86      

In [7]:
from sklearn.metrics import precision_recall_fscore_support

# Calcul global 
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

print(f"--- SCORES GLOBAUX (PONDÉRÉS) ---")
print(f"Précision globale : {precision:.4f}")
print(f"Rappel global    : {recall:.4f}")
print(f"F1-Score global  : {f1:.4f}")


--- SCORES GLOBAUX (PONDÉRÉS) ---
Précision globale : 0.9472
Rappel global    : 0.9479
F1-Score global  : 0.9468
